# Open Pipeline Risk Analysis — Q3 2026

Risk-rank all open new-business deals based on stage aging, segment exposure, competitor signals, and historical loss rates.

In [93]:
import pandas as pd
import numpy as np
from pathlib import Path

CSV_PATH = Path("/Users/guyamitai/My Drive/Open Deals at risk 2026-Q3/hubspot-crm-exports-open-newbiz-2026-07-13.csv")
df = pd.read_csv(CSV_PATH)
df.columns = df.columns.str.strip()
print(f"{len(df)} open deals loaded")
df.head(3)

328 open deals loaded


,Record ID,Deal Name,Deal Stage,Close Date,Deal owner,Amount,Deal Type,Create Date,Qualified Date,Segment,"Cumulative time in ""Business Case Confirmation (Classic)"" (HH:mm:ss)","Cumulative time in ""Business Validation (Classic)"" (HH:mm:ss)","Cumulative time in ""Demo / Presentation (Classic)"" (HH:mm:ss)","Cumulative time in ""Formal Pilot (Classic)"" (HH:mm:ss)","Cumulative time in ""Negotiation / Legal (Classic)"" (HH:mm:ss)","Cumulative time in ""SDR / Omitted Opp (Classic)"" (HH:mm:ss)",HubSpot Team,Evaluated competitors,Is Deal Closed?,Deal Tags
0,62160248969,- New Deal,SDR / Omitted Opp,2026-10-01 11:23,Adrian Sandu,NaN,New Business,2026-07-03 11:23,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SDR EMEA,NaN,False,NaN
1,61626556240,- New Deal,SDR / Omitted Opp,2026-09-24 18:36,luism@port.io,NaN,New Business,2026-06-26 18:36,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SDR US,NaN,False,NaN
2,61578697273,- New Deal,SDR / Omitted Opp,2026-09-22 22:54,Logan Loisel,NaN,New Business,2026-06-24 22:54,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SDR US,NaN,False,NaN


In [94]:
## Parse cumulative time columns into days

def hhmmss_to_days(series):
    """Convert 'HH:mm:ss' strings to fractional days."""
    def parse_one(v):
        if pd.isna(v) or v == '':
            return 0.0
        parts = str(v).split(':')
        hours = int(parts[0])
        minutes = int(parts[1]) if len(parts) > 1 else 0
        return hours / 24 + minutes / 1440
    return series.apply(parse_one)

stage_time_cols = {
    'days_in_sdr': 'Cumulative time in "SDR / Omitted  Opp (Classic)" (HH:mm:ss)',
    'days_in_demo': 'Cumulative time in "Demo / Presentation  (Classic)" (HH:mm:ss)',
    'days_in_validation': 'Cumulative time in "Business Validation (Classic)" (HH:mm:ss)',
    'days_in_pilot': 'Cumulative time in "Formal Pilot (Classic)" (HH:mm:ss)',
    'days_in_bcc': 'Cumulative time in "Business Case Confirmation (Classic)" (HH:mm:ss)',
    'days_in_legal': 'Cumulative time in "Negotiation / Legal (Classic)" (HH:mm:ss)',
}

for new_col, src_col in stage_time_cols.items():
    df[new_col] = hhmmss_to_days(df[src_col])

# Segment: use HubSpot Team to classify
def classify_segment(team):
    t = str(team).lower()
    if 'ent' in t:
        return 'Enterprise'
    elif 'mm' in t:
        return 'Mid-Market'
    else:
        return 'Other'

df['segment'] = df['HubSpot Team'].apply(classify_segment)
df['arr'] = pd.to_numeric(df['Amount'], errors='coerce').fillna(0)

# Compute average Enterprise ARR for the "Upmarket Exposure" rule
avg_ent_arr = df.loc[df['segment'] == 'Enterprise', 'arr'].mean()
print(f"Avg Enterprise ARR: ${avg_ent_arr:,.0f}")
print(df['segment'].value_counts())

Avg Enterprise ARR: $183,860
segment
Other         132
Enterprise    121
Mid-Market     75
Name: count, dtype: int64


In [95]:
## Risk Scoring Logic
# Upmarket exposure is an AMPLIFIER — only counts with a real risk flag.
# Time-in-stage signals fire based on CUMULATIVE time, regardless of current stage.
# CURRENT STAGE modifies risk: advanced stages have higher base win rates.
# Stage win rates (from historical data):
#   Negotiation/Legal → ~90%+ win | Biz Case → 84% | Pilot → 37% | Validation → 5%

# Stage progression discount: reduces signal weight based on how far the deal advanced
STAGE_DISCOUNT = {
    'SDR / Omitted  Opp': 0,       # no discount
    'SDR Discovery': 0,
    'Demo / Presentation': 0,       # 25% win rate — no discount
    'Business Validation': 0,       # 5% win rate — no discount
    'Formal Pilot': 0.3,            # 37% win rate — mild discount
    'Business Case Confirmation': 0.6,  # 84% win rate — strong discount
    'Negotiation / Legal': 0.85,    # ~90%+ win rate — near-full discount
}

def score_deal(row):
    """Return composite risk score. Stage progression reduces effective risk."""
    signals = []
    stage = str(row['Deal Stage']).strip()
    seg = row['segment']
    arr = row['arr']
    competitors = str(row.get('Evaluated competitors', '')).lower()
    tags = str(row.get('Deal Tags', '')).lower()
    owner = str(row.get('Deal owner', ''))

    # --- CRITICAL signals (weight 4) ---
    # Owner left the company — deal is orphaned
    if owner == 'Tali Cohen':
        signals.append((4, "Owner left (Tali Cohen) — deal orphaned, needs immediate reassignment"))

    if seg == 'Enterprise' and row['days_in_demo'] > 60:
        signals.append((4, f"Spent {row['days_in_demo']:.0f}d in Demo (98% loss rate)"))
    if seg == 'Enterprise' and row['days_in_validation'] > 60:
        signals.append((4, f"Spent {row['days_in_validation']:.0f}d in Validation (>80% loss rate)"))
    if seg == 'Enterprise' and row['days_in_pilot'] > 60:
        signals.append((4, f"Spent {row['days_in_pilot']:.0f}d in Pilot (>80% loss rate)"))
    if seg == 'Mid-Market' and row['days_in_sdr'] > 30:
        signals.append((4, f"Spent {row['days_in_sdr']:.0f}d in SDR (85% fail to qualify)"))
    elif seg == 'Enterprise' and row['days_in_sdr'] > 45:
        signals.append((4, f"Spent {row['days_in_sdr']:.0f}d in SDR (85% fail to qualify)"))

    # --- HIGH signals (weight 3) ---
    if 'harness' in competitors or 'harness' in tags:
        signals.append((3, "Harness competitor"))
    if row['days_in_legal'] > 90:
        signals.append((3, f"Spent {row['days_in_legal']:.0f}d in Legal (63% loss rate)"))

    # --- ELEVATED signals (weight 2) ---
    if seg == 'Enterprise' and 30 < row['days_in_demo'] <= 60:
        signals.append((2, f"Demo aging ({row['days_in_demo']:.0f}d, ~50% loss rate)"))
    if seg == 'Enterprise' and 30 < row['days_in_validation'] <= 60:
        signals.append((2, f"Validation aging ({row['days_in_validation']:.0f}d, ~40% loss rate)"))
    if seg == 'Enterprise' and 30 < row['days_in_pilot'] <= 60:
        signals.append((2, f"Pilot aging ({row['days_in_pilot']:.0f}d, ~40% loss rate)"))
    if 30 < row['days_in_legal'] <= 90:
        signals.append((2, f"Legal aging ({row['days_in_legal']:.0f}d, ~20% loss rate)"))

    # --- MODERATE signals (weight 1) ---
    if seg == 'Mid-Market' and not any(w >= 4 for w, _ in signals):
        signals.append((1, "Mid-Market volume risk (8% win rate)"))

    # --- AMPLIFIER: Upmarket exposure (only with real signals) ---
    has_real_signal = any(w >= 2 for w, _ in signals)
    if seg == 'Enterprise' and arr > avg_ent_arr and arr > 0 and has_real_signal:
        signals.append((2, f"Upmarket exposure (${arr:,.0f})"))

    # Apply stage discount: reduce effective weight based on current stage
    discount = STAGE_DISCOUNT.get(stage, 0)
    raw_weight_sum = sum(w for w, _ in signals)
    effective_weight_sum = raw_weight_sum * (1 - discount)

    signal_count = len(signals)
    composite_score = effective_weight_sum * (arr if arr > 0 else 1)

    # Risk level based on EFFECTIVE weight (after stage discount)
    max_raw_weight = max((w for w, _ in signals), default=0)
    effective_max = max_raw_weight * (1 - discount)

    if effective_max >= 3.5:
        level = 'CRITICAL'
    elif effective_max >= 2.5:
        level = 'HIGH'
    elif effective_max >= 1.5:
        level = 'ELEVATED'
    elif effective_max >= 0.5:
        level = 'MODERATE'
    else:
        level = 'LOW'

    reason_str = ' | '.join(r for _, r in signals) if signals else 'No risk signals'
    # Add stage context
    if discount > 0 and signals:
        stage_win_rates = {'Formal Pilot': '37%', 'Business Case Confirmation': '84%', 'Negotiation / Legal': '~90%+'}
        wr = stage_win_rates.get(stage, '')
        if wr:
            reason_str += f" | [Now in {stage} — {wr} historical win rate]"

    return pd.Series({
        'risk_level': level,
        'signal_count': signal_count,
        'signal_weight_sum': raw_weight_sum,
        'composite_score': composite_score,
        'risk_reasons': reason_str
    })

risk_df = df.join(df.apply(score_deal, axis=1))
risk_df = risk_df.sort_values('composite_score', ascending=False)

print(risk_df['risk_level'].value_counts())
print(f"\nTop 10 by composite score:")
print(risk_df[['Deal Name', 'arr', 'Deal Stage', 'signal_count', 'risk_level', 'risk_reasons']].head(10).to_string(index=False))

risk_level
LOW         208
MODERATE     67
CRITICAL     41
HIGH          6
ELEVATED      6
Name: count, dtype: int64

Top 10 by composite score:
                                  Deal Name       arr           Deal Stage  signal_count risk_level                                                                                                                                                                                  risk_reasons
                             Liberty Mutual 2000000.0  Business Validation             3   CRITICAL                                               Owner left (Tali Cohen) — deal orphaned, needs immediate reassignment | Spent 64d in SDR (85% fail to qualify) | Upmarket exposure ($2,000,000)
                              Goldman Sachs 1500000.0 Demo / Presentation              2   CRITICAL                                                                                        Owner left (Tali Cohen) — deal orphaned, needs immediate reassignment | Upmarket exposur

In [96]:
## Enrich with Next-Step Categories and Action Items (from account review)

# Next-step categories based on account-level blockers
NEXT_STEP_CATEGORY = {
    # Legal, Security, and Procurement Roadblocks
    'Deutsche Bank': 'Legal/Security/Procurement Roadblock',
    'Community Health Network': 'Legal/Security/Procurement Roadblock',
    # POC/POV Scoping and Execution
    'Nationwide': 'POC/POV Scoping & Execution',
    'DWP': 'POC/POV Scoping & Execution',
    'GardaWorld': 'POC/POV Scoping & Execution',
    'Janus Henderson': 'POC/POV Scoping & Execution',
    'Laerdal Medical': 'POC/POV Scoping & Execution',
    # Executive Alignment & Business Cases
    'Manulife': 'Executive Alignment & Business Case',
    'GFT Technologies': 'Executive Alignment & Business Case',
    'Nykredit': 'Executive Alignment & Business Case',
    'Cresta': 'Executive Alignment & Business Case',
    'Direct Supply': 'Executive Alignment & Business Case',
    'Hansen Technologies': 'Executive Alignment & Business Case',
    # Chasing Unresponsive Accounts (Ghosting)
    'Bosch Brazil': 'Ghosting — Unresponsive',
    'Garmin': 'Ghosting — Unresponsive',
    'San Disk': 'Ghosting — Unresponsive',
    'PNC': 'Ghosting — Unresponsive',
}

# Recommended action items per deal
ACTION_ITEMS = {
    # 1. Quick-Start Templates — perceived complexity / lost to Cortex
    'GoFundMe': 'Create Quick-Start Templates (lost to Cortex — simplify onboarding)',
    'Smartsheet': 'Create Quick-Start Templates (lost to Cortex — simplify onboarding)',
    'Direct Supply': 'Create Quick-Start Templates + Prove ROI (perceived complexity & ROI justification)',
    'Woolworths': 'Create Quick-Start Templates (stalled — perceived complexity)',
    # 2. Prove ROI and Involve Execs Early
    'Cresta': 'Prove ROI & Involve Execs Early (ROI justification needed)',
    'Hansen Technologies': 'Prove ROI & Involve Execs Early (ROI justification needed)',
    'Schneider Electric': 'Prove ROI & Involve Execs Early (budget freeze/reallocation)',
    'STIME': 'Prove ROI & Involve Execs Early (budget freeze/reallocation)',
    'Zuora': 'Prove ROI & Involve Execs Early (budget freeze/reallocation)',
    'Garmin': 'Prove ROI & Involve Execs Early (cost/pricing pushback)',
    'Community Health Network': 'Prove ROI + Fast-Track Security Package (cost pushback & TPRA stall)',
    'Close Brothers': 'Prove ROI & Involve Execs Early (cost/pricing pushback)',
    'Manulife': 'Prove ROI & Involve Execs Early (exec alignment call needed)',
    'GFT Technologies': 'Prove ROI & Involve Execs Early (exec alignment call needed)',
    'Nykredit': 'Prove ROI & Involve Execs Early (exec alignment call needed)',
    # 3. Fast-Track Security Package
    'Deutsche Bank': 'Fast-Track Security Package (blocked by 1,200-question Coupa assessment)',
}

# Fuzzy match deal names (substring match to handle suffixes like "- Renewal")
def match_deal(deal_name, lookup):
    for key, val in lookup.items():
        if key.lower() in deal_name.lower():
            return val
    return None

risk_df['next_step_category'] = risk_df['Deal Name'].apply(lambda d: match_deal(d, NEXT_STEP_CATEGORY) or '—')
risk_df['action_item'] = risk_df['Deal Name'].apply(lambda d: match_deal(d, ACTION_ITEMS) or '—')

# Summary
categorized = risk_df[risk_df['next_step_category'] != '—']
print(f"Deals matched to a next-step category: {len(categorized)} / {len(risk_df)}")
print(f"Deals matched to an action item: {(risk_df['action_item'] != '—').sum()} / {len(risk_df)}")
print(f"\nCategory breakdown:")
print(categorized['next_step_category'].value_counts().to_string())


Deals matched to a next-step category: 12 / 328
Deals matched to an action item: 9 / 328

Category breakdown:
next_step_category
POC/POV Scoping & Execution             6
Executive Alignment & Business Case     3
Legal/Security/Procurement Roadblock    2
Ghosting — Unresponsive                 1


### Risk Factors Reference

| Weight | Level | Signal | Condition | Threshold |
|--------|-------|--------|-----------|-----------|
| 4 | CRITICAL | Deal orphaned | Owner left the company | Owner = Tali Cohen |
| 4 | CRITICAL | Stuck in Demo | Enterprise, cumulative days in Demo | >60 days (98% loss rate) |
| 4 | CRITICAL | Stuck in Validation | Enterprise, cumulative days in Validation | >60 days (>80% loss rate) |
| 4 | CRITICAL | Stuck in Pilot | Enterprise, cumulative days in Pilot | >60 days (>80% loss rate) |
| 4 | CRITICAL | Stuck in SDR | Mid-Market cumulative days in SDR | >30 days (85% fail to qualify) |
| 4 | CRITICAL | Stuck in SDR | Enterprise cumulative days in SDR | >45 days (85% fail to qualify) |
| 3 | HIGH | Harness competitor | Harness in evaluated competitors or deal tags | Present |
| 3 | HIGH | Stuck in Legal | Cumulative days in Legal | >90 days (63% loss rate) |
| 2 | ELEVATED | Demo aging | Enterprise, cumulative days in Demo | 30–60 days (~50% loss rate) |
| 2 | ELEVATED | Validation aging | Enterprise, cumulative days in Validation | 30–60 days (~40% loss rate) |
| 2 | ELEVATED | Pilot aging | Enterprise, cumulative days in Pilot | 30–60 days (~40% loss rate) |
| 2 | ELEVATED | Legal aging | Cumulative days in Legal | 30–90 days (~20% loss rate) |
| 1 | MODERATE | Mid-Market volume risk | Mid-Market segment, no critical signals | Always (8% win rate) |
| 2 | AMPLIFIER | Upmarket exposure | Enterprise, ARR > avg Enterprise ARR | Only fires with a weight≥2 signal |

**Stage progression discount** reduces effective signal weight based on current stage:
- Formal Pilot (37% win rate): 30% discount
- Business Case Confirmation (84% win rate): 60% discount
- Negotiation / Legal (~90%+ win rate): 85% discount

---
## Output 1: All Deals — Risk Status & Explanation

In [97]:
## OUTPUT 1: All deals with risk status

all_deals = risk_df[['Deal Name', 'Deal Stage', 'Deal owner', 'HubSpot Team', 'segment',
                     'arr', 'risk_level', 'signal_count', 'next_step_category', 'action_item', 'risk_reasons']].copy()
all_deals['arr_fmt'] = all_deals['arr'].apply(lambda x: f"${x:,.0f}" if x > 0 else '-')

# Order by risk level: CRITICAL → HIGH → ELEVATED → MODERATE → LOW
level_order = {'CRITICAL': 0, 'HIGH': 1, 'ELEVATED': 2, 'MODERATE': 3, 'LOW': 4}
all_deals['_level_sort'] = all_deals['risk_level'].map(level_order)
all_deals = all_deals.sort_values(['_level_sort', 'arr'], ascending=[True, False]).drop(columns='_level_sort')

print(f"Total open deals: {len(all_deals)}")
print(f"\nRisk distribution:")
for level in ['CRITICAL', 'HIGH', 'ELEVATED', 'MODERATE', 'LOW']:
    subset = all_deals[all_deals['risk_level'] == level]
    if len(subset) > 0:
        print(f"  {level:10s} — {len(subset):3d} deals | ARR: ${subset['arr'].sum():,.0f}")

print(f"\n{'='*100}")
all_deals[['Deal Name', 'arr_fmt', 'Deal owner', 'Deal Stage', 'risk_level', 'next_step_category', 'action_item', 'risk_reasons']]

Total open deals: 328

Risk distribution:
  CRITICAL   —  41 deals | ARR: $9,498,000
  HIGH       —   6 deals | ARR: $1,945,000
  ELEVATED   —   6 deals | ARR: $1,540,000
  MODERATE   —  67 deals | ARR: $7,283,700
  LOW        — 208 deals | ARR: $10,221,155



,Deal Name,arr_fmt,Deal owner,Deal Stage,risk_level,next_step_category,action_item,risk_reasons
189,Liberty Mutual,"$2,000,000",Tali Cohen,Business Validation,CRITICAL,—,—,"Owner left (Tali Cohen) — deal orphaned, needs..."
133,Goldman Sachs,"$1,500,000",Tali Cohen,Demo / Presentation,CRITICAL,—,—,"Owner left (Tali Cohen) — deal orphaned, needs..."
233,Optum Inc,"$500,000",Rick Walker,Demo / Presentation,CRITICAL,—,—,Spent 53d in SDR (85% fail to qualify) | Upmar...
187,Lego,"$350,000",James Pritchard,Demo / Presentation,CRITICAL,—,—,Spent 45d in SDR (85% fail to qualify) | Upmar...
242,PayPal,"$300,000",Jeff Graham,Business Validation,CRITICAL,—,—,Spent 99d in Pilot (>80% loss rate) | Harness ...
...,...,...,...,...,...,...,...,...
126,G-P/Globalization Partners,-,Aaron Marans,SDR / Omitted Opp,LOW,—,—,No risk signals
131,Giesecke+Devrient,-,John Barnes,SDR / Omitted Opp,LOW,—,—,No risk signals
134,Grupo DPSP,-,Logan Loisel,SDR / Omitted Opp,LOW,—,—,No risk signals
137,Hallmark,-,Dylan Leija,SDR / Omitted Opp,LOW,—,—,No risk signals


---
## Output 2: Top 6 Deals at Highest Risk

In [98]:
## OUTPUT 2: Top 6 highest-risk deals
# Excludes Negotiation/Legal deals (90%+ historical win rate — not truly at risk)

top_risk = risk_df[
    (risk_df['signal_count'] > 0) &
    (risk_df['Deal Stage'] != 'Negotiation / Legal')
].sort_values('composite_score', ascending=False).head(6)

print("🔴 TOP 6 DEALS AT HIGHEST RISK\n")
for i, (_, row) in enumerate(top_risk.iterrows(), 1):
    arr_str = f"${row['arr']:,.0f}" if row['arr'] > 0 else 'No ARR'
    print(f"  {i}. {row['Deal Name']} — {arr_str}")
    print(f"     Owner: {row['Deal owner']} | Stage: {row['Deal Stage']}")
    print(f"     Risk: {row['risk_reasons']}")
    print()

🔴 TOP 6 DEALS AT HIGHEST RISK

  1. Liberty Mutual — $2,000,000
     Owner: Tali Cohen | Stage: Business Validation
     Risk: Owner left (Tali Cohen) — deal orphaned, needs immediate reassignment | Spent 64d in SDR (85% fail to qualify) | Upmarket exposure ($2,000,000)

  2. Goldman Sachs — $1,500,000
     Owner: Tali Cohen | Stage: Demo / Presentation 
     Risk: Owner left (Tali Cohen) — deal orphaned, needs immediate reassignment | Upmarket exposure ($1,500,000)

  3. Optum Inc — $500,000
     Owner: Rick Walker | Stage: Demo / Presentation 
     Risk: Spent 53d in SDR (85% fail to qualify) | Upmarket exposure ($500,000)

  4. DWP — $500,000
     Owner: James Pritchard | Stage: Formal Pilot
     Risk: Spent 140d in Validation (>80% loss rate) | Demo aging (43d, ~50% loss rate) | Upmarket exposure ($500,000) | [Now in Formal Pilot — 37% historical win rate]

  5. PayPal  — $300,000
     Owner: Jeff Graham | Stage: Business Validation
     Risk: Spent 99d in Pilot (>80% loss rate) | 

---
## Output 3: Top 6 Deals We Can Save — Act Now to Increase ARR Wins

In [99]:
## OUTPUT 3: Top 6 deals we can save if we act now
# Saveable deals: in active stages (Validation, Pilot, Biz Case) — NOT Legal.
# Two types:
# 1. ELEVATED/HIGH — approaching danger, intervention prevents them from tipping
# 2. "Advanced survivors" — CRITICAL but made it to Biz Case despite past slowness

saveable = risk_df[
    (risk_df['signal_count'] >= 1) &
    (risk_df['arr'] > 0) &
    (risk_df['Deal Stage'].isin(['Business Validation', 'Formal Pilot',
                                  'Business Case Confirmation'])) &
    (
        # Type 1: Elevated/High — approaching danger
        (risk_df['risk_level'].isin(['ELEVATED', 'HIGH'])) |
        # Type 2: Critical BUT already advanced to Biz Case (survivors)
        ((risk_df['risk_level'] == 'CRITICAL') &
         (risk_df['Deal Stage'] == 'Business Case Confirmation'))
    )
].copy()

saveable = saveable.sort_values('composite_score', ascending=False).head(6)

def suggest_action(row):
    """Data-driven action based on stage history, competitors, team, and owner."""
    stage = row['Deal Stage']
    competitors = str(row.get('Evaluated competitors', '')).strip()
    team = str(row.get('HubSpot Team', ''))
    owner = str(row.get('Deal owner', ''))
    actions = []

    # Tali Cohen left — her deals need new ownership
    if owner == 'Tali Cohen':
        actions.append("OWNER LEFT: Reassign immediately, re-engage champion before momentum dies")
        return ' | '.join(actions)

    # Competitor-specific
    if 'harness' in competitors.lower():
        actions.append("Harness active — run competitive displacement, exec-to-exec, differentiation demo")
    elif competitors and competitors.lower() not in ['nan', '']:
        comp_names = competitors.split(';')[0].strip()
        actions.append(f"Evaluating {comp_names} — sharpen differentiators, validate why Port wins")

    # Stage-specific + history-aware
    if stage == 'Business Case Confirmation':
        if row['days_in_pilot'] > 60 or row['days_in_validation'] > 60:
            actions.append("Slow history but made it here — lock ROI now before deal loses momentum")
        else:
            actions.append("Align on ROI, secure budget holder sign-off, push to legal")
    elif stage == 'Business Validation':
        if row['days_in_demo'] > 30:
            actions.append("Long Demo phase — propose fast-track pilot with clear success criteria")
        else:
            actions.append("Confirm success criteria, remove blockers, secure champion")
    elif stage == 'Formal Pilot':
        if row['days_in_validation'] > 40:
            actions.append("Slow validation — ensure pilot has tight timeline & measurable outcomes")
        else:
            actions.append("Drive pilot to success — weekly check-ins, prep business case early")

    # US Ent high-value → exec attention
    if 'US Ent' in team and row['arr'] >= 250000:
        actions.append("US Ent high-value — consider exec sponsor involvement")

    return ' | '.join(actions) if actions else "Review deal plan with owner — identify stall point"

print("🟢 TOP 6 DEALS WE CAN SAVE — ACT NOW\n")
for i, (_, row) in enumerate(saveable.iterrows(), 1):
    arr_str = f"${row['arr']:,.0f}"
    survivor = " [ADVANCED SURVIVOR]" if row['risk_level'] == 'CRITICAL' else ""
    cat = row.get('next_step_category', '—')
    action_item = row.get('action_item', '—')
    print(f"  {i}. {row['Deal Name']} — {arr_str}{survivor}")
    print(f"     Owner: {row['Deal owner']} | Team: {row['HubSpot Team']} | Stage: {row['Deal Stage']}")
    if cat != '—':
        print(f"     Category: {cat}")
    if action_item != '—':
        print(f"     Recommended Action Item: {action_item}")
    print(f"     Why at risk: {row['risk_reasons']}")
    print(f"     Suggested Next Step: {suggest_action(row)}")
    print()

🟢 TOP 6 DEALS WE CAN SAVE — ACT NOW

  1. DWP — $500,000
     Owner: James Pritchard | Team: EMEA Ent | Stage: Formal Pilot
     Category: POC/POV Scoping & Execution
     Why at risk: Spent 140d in Validation (>80% loss rate) | Demo aging (43d, ~50% loss rate) | Upmarket exposure ($500,000) | [Now in Formal Pilot — 37% historical win rate]
     Suggested Next Step: Evaluating Backstage — sharpen differentiators, validate why Port wins | Slow validation — ensure pilot has tight timeline & measurable outcomes

  2. GFT Technologies - IDP - 13k Users — $400,000
     Owner: John Barnes | Team: EMEA Ent | Stage: Business Validation
     Category: Executive Alignment & Business Case
     Recommended Action Item: Prove ROI & Involve Execs Early (exec alignment call needed)
     Why at risk: Harness competitor | Upmarket exposure ($400,000)
     Suggested Next Step: Harness active — run competitive displacement, exec-to-exec, differentiation demo | Confirm success criteria, remove blockers, s

In [100]:
## EXPORT all outputs
import io, sys

# 1. CSV — full deal table with risk scores, categories, action items
export_cols = ['Deal Name', 'Deal Stage', 'Deal owner', 'HubSpot Team', 'segment',
               'arr', 'risk_level', 'signal_count', 'composite_score',
               'next_step_category', 'action_item', 'risk_reasons']
risk_df[export_cols].to_csv('open_pipe_risk_output.csv', index=False)

# 2. Text — capture printed outputs from OUTPUT 1/2/3
buf = io.StringIO()

buf.write("=" * 100 + "\n")
buf.write("OPEN PIPELINE RISK ANALYSIS — FULL OUTPUT\n")
buf.write("=" * 100 + "\n\n")

# Risk distribution
buf.write(f"Total open deals: {len(risk_df)}\n\nRisk distribution:\n")
for level in ['CRITICAL', 'HIGH', 'ELEVATED', 'MODERATE', 'LOW']:
    subset = risk_df[risk_df['risk_level'] == level]
    if len(subset) > 0:
        buf.write(f"  {level:10s} — {len(subset):3d} deals | ARR: ${subset['arr'].sum():,.0f}\n")

# Top 6 highest risk
buf.write(f"\n{'=' * 100}\nTOP 6 HIGHEST-RISK DEALS\n{'=' * 100}\n\n")
top6 = risk_df.head(6)
for i, (_, row) in enumerate(top6.iterrows(), 1):
    arr_str = f"${row['arr']:,.0f}" if row['arr'] > 0 else '-'
    buf.write(f"  {i}. {row['Deal Name']} — {arr_str} | {row['risk_level']}\n")
    buf.write(f"     Owner: {row['Deal owner']} | Stage: {row['Deal Stage']}\n")
    if row.get('next_step_category', '—') != '—':
        buf.write(f"     Category: {row['next_step_category']}\n")
    if row.get('action_item', '—') != '—':
        buf.write(f"     Action Item: {row['action_item']}\n")
    buf.write(f"     Risk: {row['risk_reasons']}\n\n")

# Top 6 saveable
buf.write(f"{'=' * 100}\nTOP 6 DEALS WE CAN SAVE — ACT NOW\n{'=' * 100}\n\n")
saveable_export = risk_df[
    (risk_df['signal_count'] >= 1) & (risk_df['arr'] > 0) &
    (risk_df['Deal Stage'].isin(['Business Validation', 'Formal Pilot', 'Business Case Confirmation'])) &
    ((risk_df['risk_level'].isin(['ELEVATED', 'HIGH'])) |
     ((risk_df['risk_level'] == 'CRITICAL') & (risk_df['Deal Stage'] == 'Business Case Confirmation')))
].sort_values('composite_score', ascending=False).head(6)

for i, (_, row) in enumerate(saveable_export.iterrows(), 1):
    arr_str = f"${row['arr']:,.0f}"
    survivor = " [ADVANCED SURVIVOR]" if row['risk_level'] == 'CRITICAL' else ""
    buf.write(f"  {i}. {row['Deal Name']} — {arr_str}{survivor}\n")
    buf.write(f"     Owner: {row['Deal owner']} | Team: {row['HubSpot Team']} | Stage: {row['Deal Stage']}\n")
    if row.get('next_step_category', '—') != '—':
        buf.write(f"     Category: {row['next_step_category']}\n")
    if row.get('action_item', '—') != '—':
        buf.write(f"     Action Item: {row['action_item']}\n")
    buf.write(f"     Risk: {row['risk_reasons']}\n")
    buf.write(f"     Next Step: {suggest_action(row)}\n\n")

with open('open_pipe_risk_output.txt', 'w') as f:
    f.write(buf.getvalue())

print("Exported:")
print(f"  1. open_pipe_risk_output.csv  — full table ({len(risk_df)} rows x {len(export_cols)} cols)")
print(f"  2. open_pipe_risk_output.txt  — formatted text summary")
import os
print(f"\nFiles saved to: {os.getcwd()}")


Exported:
  1. open_pipe_risk_output.csv  — full table (328 rows x 12 cols)
  2. open_pipe_risk_output.txt  — formatted text summary

Files saved to: /Users/guyamitai/Documents/Snow/snowflake-guy


In [101]:
## OUTPUT 4: Per-company summary table

# Determine "highest risk" — top 6 by composite score (excluding Legal stage)
top_risk_names = set(
    risk_df[
        (risk_df['signal_count'] > 0) &
        (risk_df['Deal Stage'] != 'Negotiation / Legal')
    ].sort_values('composite_score', ascending=False).head(6)['Deal Name']
)

# Determine "act now" — saveable deals (active stages, elevated+)
saveable_names = set(
    risk_df[
        (risk_df['signal_count'] >= 1) & (risk_df['arr'] > 0) &
        (risk_df['Deal Stage'].isin(['Business Validation', 'Formal Pilot', 'Business Case Confirmation'])) &
        ((risk_df['risk_level'].isin(['ELEVATED', 'HIGH'])) |
         ((risk_df['risk_level'] == 'CRITICAL') & (risk_df['Deal Stage'] == 'Business Case Confirmation')))
    ]['Deal Name']
)

# Build summary
summary_rows = []
for _, row in risk_df.iterrows():
    if row['signal_count'] == 0:
        continue  # skip no-signal deals

    act_now = 'Yes' if row['Deal Name'] in saveable_names else 'No'
    is_highest_risk = 'Yes' if row['Deal Name'] in top_risk_names else 'No'

    # Action needed: use the enrichment action_item if available, otherwise the suggest_action logic
    action = row['action_item'] if row['action_item'] != '—' else suggest_action(row)

    # Why action is needed: the next_step_category context if available, otherwise derived from risk reasons
    if row['next_step_category'] != '—':
        why_action = row['next_step_category']
    elif row['Deal owner'] == 'Tali Cohen':
        why_action = 'Deal owner left company — orphaned deal'
    elif 'harness' in str(row.get('Evaluated competitors', '')).lower():
        why_action = 'Active competitor threat (Harness)'
    elif row['days_in_pilot'] > 60 or row['days_in_validation'] > 60:
        why_action = 'Stalled too long in stage — momentum dying'
    elif row['days_in_demo'] > 30:
        why_action = 'Aging in early stage — risk of disengagement'
    else:
        why_action = 'Time-based risk signal triggered'

    summary_rows.append({
        'Company': row['Deal Name'],
        'ARR': f"${row['arr']:,.0f}" if row['arr'] > 0 else '-',
        'Risk Level': row['risk_level'],
        'Why at Risk': row['risk_reasons'],
        'Act Now?': act_now,
        'Action Needed': action,
        'Why Action Needed': why_action,
        'Highest Risk?': is_highest_risk,
    })

summary_df = pd.DataFrame(summary_rows)
print(f"Deals with risk signals: {len(summary_df)}")
summary_df

Deals with risk signals: 129


,Company,ARR,Risk Level,Why at Risk,Act Now?,Action Needed,Why Action Needed,Highest Risk?
0,Liberty Mutual,"$2,000,000",CRITICAL,"Owner left (Tali Cohen) — deal orphaned, needs...",No,"OWNER LEFT: Reassign immediately, re-engage ch...",Deal owner left company — orphaned deal,Yes
1,Goldman Sachs,"$1,500,000",CRITICAL,"Owner left (Tali Cohen) — deal orphaned, needs...",No,"OWNER LEFT: Reassign immediately, re-engage ch...",Deal owner left company — orphaned deal,Yes
2,Deutsche Bank,"$1,900,000",MODERATE,Spent 78d in Pilot (>80% loss rate) | Harness ...,No,"Fast-Track Security Package (blocked by 1,200-...",Legal/Security/Procurement Roadblock,No
3,Optum Inc,"$500,000",CRITICAL,Spent 53d in SDR (85% fail to qualify) | Upmar...,No,US Ent high-value — consider exec sponsor invo...,Time-based risk signal triggered,Yes
4,DWP,"$500,000",HIGH,Spent 140d in Validation (>80% loss rate) | De...,Yes,Evaluating Backstage — sharpen differentiators...,POC/POV Scoping & Execution,Yes
...,...,...,...,...,...,...,...,...
124,Distyl AI,-,MODERATE,Mid-Market volume risk (8% win rate),No,Review deal plan with owner — identify stall p...,Time-based risk signal triggered,No
125,Versant Media,-,MODERATE,Mid-Market volume risk (8% win rate),No,Review deal plan with owner — identify stall p...,Time-based risk signal triggered,No
126,Amdocs,-,MODERATE,Mid-Market volume risk (8% win rate),No,Review deal plan with owner — identify stall p...,Time-based risk signal triggered,No
127,Tangerine,-,MODERATE,Mid-Market volume risk (8% win rate),No,Review deal plan with owner — identify stall p...,Time-based risk signal triggered,No


In [102]:
## Risk Factors Reference Table

risk_factors = pd.DataFrame([
    {'Weight': 4, 'Level': 'CRITICAL', 'Signal': 'Deal orphaned', 'Condition': 'Owner left the company (Tali Cohen)', 'Threshold': 'Owner match'},
    {'Weight': 4, 'Level': 'CRITICAL', 'Signal': 'Stuck in Demo', 'Condition': 'Enterprise, cumulative days in Demo', 'Threshold': '>60 days (98% loss rate)'},
    {'Weight': 4, 'Level': 'CRITICAL', 'Signal': 'Stuck in Validation', 'Condition': 'Enterprise, cumulative days in Validation', 'Threshold': '>60 days (>80% loss rate)'},
    {'Weight': 4, 'Level': 'CRITICAL', 'Signal': 'Stuck in Pilot', 'Condition': 'Enterprise, cumulative days in Pilot', 'Threshold': '>60 days (>80% loss rate)'},
    {'Weight': 4, 'Level': 'CRITICAL', 'Signal': 'Stuck in SDR', 'Condition': 'Mid-Market cumulative days in SDR', 'Threshold': '>30 days (85% fail to qualify)'},
    {'Weight': 4, 'Level': 'CRITICAL', 'Signal': 'Stuck in SDR', 'Condition': 'Enterprise cumulative days in SDR', 'Threshold': '>45 days (85% fail to qualify)'},
    {'Weight': 3, 'Level': 'HIGH', 'Signal': 'Harness competitor', 'Condition': 'Harness in evaluated competitors or deal tags', 'Threshold': 'Present'},
    {'Weight': 3, 'Level': 'HIGH', 'Signal': 'Stuck in Legal', 'Condition': 'Cumulative days in Legal', 'Threshold': '>90 days (63% loss rate)'},
    {'Weight': 2, 'Level': 'ELEVATED', 'Signal': 'Demo aging', 'Condition': 'Enterprise, cumulative days in Demo', 'Threshold': '30–60 days (~50% loss rate)'},
    {'Weight': 2, 'Level': 'ELEVATED', 'Signal': 'Validation aging', 'Condition': 'Enterprise, cumulative days in Validation', 'Threshold': '30–60 days (~40% loss rate)'},
    {'Weight': 2, 'Level': 'ELEVATED', 'Signal': 'Pilot aging', 'Condition': 'Enterprise, cumulative days in Pilot', 'Threshold': '30–60 days (~40% loss rate)'},
    {'Weight': 2, 'Level': 'ELEVATED', 'Signal': 'Legal aging', 'Condition': 'Cumulative days in Legal', 'Threshold': '30–90 days (~20% loss rate)'},
    {'Weight': 1, 'Level': 'MODERATE', 'Signal': 'Mid-Market volume risk', 'Condition': 'Mid-Market segment, no critical signals', 'Threshold': 'Always (8% win rate)'},
    {'Weight': 2, 'Level': 'AMPLIFIER', 'Signal': 'Upmarket exposure', 'Condition': 'Enterprise, ARR > avg Enterprise ARR', 'Threshold': 'Only fires with weight≥2 signal'},
])

print("Stage Progression Discount (reduces effective signal weight):")
print("  Formal Pilot (37% win rate): 30% discount")
print("  Business Case Confirmation (84% win rate): 60% discount")
print("  Negotiation / Legal (~90%+ win rate): 85% discount")
print()
risk_factors

Stage Progression Discount (reduces effective signal weight):
  Formal Pilot (37% win rate): 30% discount
  Business Case Confirmation (84% win rate): 60% discount
  Negotiation / Legal (~90%+ win rate): 85% discount



,Weight,Level,Signal,Condition,Threshold
0,4,CRITICAL,Deal orphaned,Owner left the company (Tali Cohen),Owner match
1,4,CRITICAL,Stuck in Demo,"Enterprise, cumulative days in Demo",>60 days (98% loss rate)
2,4,CRITICAL,Stuck in Validation,"Enterprise, cumulative days in Validation",>60 days (>80% loss rate)
3,4,CRITICAL,Stuck in Pilot,"Enterprise, cumulative days in Pilot",>60 days (>80% loss rate)
4,4,CRITICAL,Stuck in SDR,Mid-Market cumulative days in SDR,>30 days (85% fail to qualify)
5,4,CRITICAL,Stuck in SDR,Enterprise cumulative days in SDR,>45 days (85% fail to qualify)
6,3,HIGH,Harness competitor,Harness in evaluated competitors or deal tags,Present
7,3,HIGH,Stuck in Legal,Cumulative days in Legal,>90 days (63% loss rate)
8,2,ELEVATED,Demo aging,"Enterprise, cumulative days in Demo",30–60 days (~50% loss rate)
9,2,ELEVATED,Validation aging,"Enterprise, cumulative days in Validation",30–60 days (~40% loss rate)
